# Polygon.io — Data Ingestion

Pulls **2 years** of daily bars for stocks, options, and futures.  
Rate-limited to **5 req/min** (free tier) with automatic retry (exponential back-off) and **checkpoint/resume** (existing parquet files are skipped).

```
data/raw/
  stocks/           {TICKER}_day.parquet
  options/
    snapshots/      {UNDERLYING}_{DATE}.parquet   <- contract reference (ticker, strike, expiry)
    history/        {CONTRACT}_day.parquet        <- OHLCV
  futures/          {CONTRACT}_day.parquet
```

> **Note:** Greeks, IV, and open interest require the Starter plan ($29/mo) and are not fetched here.

In [1]:
import os, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv

# Running under VS Code (local kernel) inherits VSCODE_* env vars — treat as a
# local dev session even if /content happens to exist, so we never clobber the
# repo file with a redundant clone/Drive mount.
IN_VSCODE = bool(os.environ.get('VSCODE_PID') or os.environ.get('VSCODE_CWD'))
IN_COLAB  = ('google.colab' in sys.modules or os.path.exists('/content')) and not IN_VSCODE

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    _token = userdata.get('GITHUB_TOKEN')
    _repo  = 'shreyasnat2804/JEPA-quant'
    _dest  = '/content/JEPA-quant'

    # Pull the latest repo code (.py modules, etc.) into the VM.
    # NOTE: this does NOT refresh the notebook you're currently viewing — a cell
    # can't reload its own tab. To get the latest notebook, reopen it via
    # File -> Open notebook -> GitHub tab (or Runtime -> revert if editing in Colab).
    if not os.path.exists(_dest):
        subprocess.run(
            ['git', 'clone', f'https://{_token}@github.com/{_repo}.git', _dest],
            check=True,
        )
    else:
        subprocess.run(['git', '-C', _dest, 'pull'], check=True)

    os.chdir(f'{_dest}/notebooks')
    print(f'Repo synced. Working directory: {os.getcwd()}')
else:
    load_dotenv('../.env')
    where = 'VS Code' if IN_VSCODE else 'local'
    print(f'Running in {where} — skipping Drive mount and git pull')


Running in VS Code — skipping Drive mount and git pull


In [ ]:
# Hot-reload edited jepa_quant/*.py modules without restarting the kernel.
# After you push from VS Code and re-run the setup cell's `git pull`, autoreload
# swaps the new code in on the next cell execution — no kernel restart, no lost state.
%load_ext autoreload
%autoreload 2

In [2]:
%pip install aiohttp nest-asyncio python-dotenv tqdm pandas pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import asyncio
import logging
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Optional

import aiohttp
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

nest_asyncio.apply()  # allow asyncio.run() inside Jupyter

# Colab: falls back to Colab Secrets (add MARKET_DATA_API_KEY in the 🔑 tab)
# Local: already loaded from .env in the setup cell above
load_dotenv('../.env')

if not os.environ.get('MARKET_DATA_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['MARKET_DATA_API_KEY'] = userdata.get('MARKET_DATA_API_KEY')
    except Exception:
        raise EnvironmentError(
            'MARKET_DATA_API_KEY not found.\n'
            '  Local: add it to JEPA-quant/.env\n'
            '  Colab: Secrets tab (🔑) → add MARKET_DATA_API_KEY'
        )

API_KEY    = os.environ['MARKET_DATA_API_KEY']
BASE_URL   = 'https://api.polygon.io'
RATE_LIMIT = 0.08  # 5 requests/minute (free tier)

END_DATE   = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=730)).isoformat()

# Colab writes to Drive; local runs write to data/raw/ in the project root
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/JEPA-QUANT/data/raw')
else:
    DATA_DIR = Path('../data/raw')

for sub in ('stocks', 'options/snapshots', 'options/history', 'futures'):
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('polygon_ingest')

print(f'Date range: {START_DATE}  ->  {END_DATE}')
print(f'Data directory: {DATA_DIR}')

Date range: 2024-06-03  ->  2026-06-03
Data directory: ../data/raw


/Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class TokenBucket:
    """Async token-bucket rate limiter."""

    def __init__(self, rate: float):
        self.rate = rate
        self.tokens = float(rate)
        self._last = time.monotonic()
        self._lock = asyncio.Lock()

    async def acquire(self) -> None:
        async with self._lock:
            now = time.monotonic()
            self.tokens = min(self.rate, self.tokens + (now - self._last) * self.rate)
            self._last = now
            if self.tokens < 1.0:
                await asyncio.sleep((1.0 - self.tokens) / self.rate)
                self.tokens = 0.0
            else:
                self.tokens -= 1.0

In [5]:
import ssl, certifi  # macOS python.org builds don't trust the system keychain


class PolygonClient:
    _MAX_RETRIES = 5

    def __init__(self, api_key: str, rate: float = 5.0):
        self.api_key = api_key
        self._limiter = TokenBucket(rate)
        self._session: Optional[aiohttp.ClientSession] = None
        # Explicit certifi trust store — fixes CERTIFICATE_VERIFY_FAILED on macOS,
        # no-op on Colab/Linux. See CLAUDE.md (notebook env gotcha).
        self._ssl_ctx = ssl.create_default_context(cafile=certifi.where())

    async def _get_session(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(
                headers={'Authorization': f'Bearer {self.api_key}'},
                timeout=aiohttp.ClientTimeout(total=30),
                connector=aiohttp.TCPConnector(ssl=self._ssl_ctx),
            )
        return self._session

    async def close(self) -> None:
        if self._session and not self._session.closed:
            await self._session.close()

    async def get(self, url: str, params: Optional[dict] = None) -> dict:
        session = await self._get_session()
        for attempt in range(self._MAX_RETRIES):
            await self._limiter.acquire()
            try:
                async with session.get(url, params=params) as resp:
                    if resp.status == 429:
                        # Sleep out the full minute window — exponential backoff
                        # wastes requests retrying before the limit resets.
                        log.warning('429 rate-limit; sleeping 61s')
                        await asyncio.sleep(61)
                        continue
                    resp.raise_for_status()
                    return await resp.json()
            except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                if attempt == self._MAX_RETRIES - 1:
                    raise
                wait = 2 ** attempt
                log.warning('Error (attempt %d): %s -- retry in %ds', attempt + 1, exc, wait)
                await asyncio.sleep(wait)
        return {}

    async def paginate(self, url: str, params: Optional[dict] = None) -> list:
        """Follow Polygon next_url cursors to collect all pages."""
        all_results = []
        first = True
        while url:
            data = await self.get(url, params if first else None)
            first = False
            all_results.extend(data.get('results') or [])
            url = data.get('next_url', '')
        return all_results


client = PolygonClient(API_KEY, RATE_LIMIT)
print('Client ready.')

Client ready.


## 1 - Stocks

One paginated request per ticker — 2 years of daily bars fit in a single call (Polygon returns up to 50 000 bars per page).  
Existing `.parquet` files are skipped on re-runs.

In [6]:
STOCK_TICKERS = [
    # Core large-cap universe -- edit to match your strategy
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA',
    'JPM',  'V',    'MA',   'BAC',  'GS',    'MS',
    'XOM',  'CVX',  'COP',
    'LLY',  'UNH',  'PFE',  'MRK',  'ABBV',
    'AVGO', 'AMD',  'INTC', 'QCOM', 'TXN',
    'HD',   'WMT',  'COST', 'TGT',
    'CAT',  'RTX',  'HON',  'DE',
    'NEE',  'DUK',  'SO',
    # Broad ETFs
    'SPY',  'QQQ',  'IWM',  'DIA',  'GLD',  'SLV',  'USO',  'TLT',
]
STOCK_TICKERS = sorted(set(STOCK_TICKERS))
print(f'{len(STOCK_TICKERS)} tickers in universe')

45 tickers in universe


In [7]:
_OHLCV_COLS = ['ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions']


def _atomic_to_parquet(df: pd.DataFrame, path: Path) -> None:
    """Write then rename, so an interrupt can't leave a half-written parquet."""
    tmp = path.with_suffix('.parquet.tmp')
    df.to_parquet(tmp)
    tmp.replace(path)


def _empty_ohlcv(ticker: str, path: Path) -> pd.DataFrame:
    df = pd.DataFrame(columns=_OHLCV_COLS)
    df.index.name = 'ts'
    _atomic_to_parquet(df, path)
    return df


async def fetch_stock_bars(
    ticker: str,
    timespan: str = 'day',
    multiplier: int = 1,
) -> pd.DataFrame:
    out = DATA_DIR / 'stocks' / f'{ticker}_{timespan}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{ticker}/range'
        f'/{multiplier}/{timespan}/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'true', 'sort': 'asc'})
    if not rows:
        log.warning('No bars: %s — saving empty result', ticker)
        return _empty_ohlcv(ticker, out)

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = ticker
    df = df.set_index('ts')[_OHLCV_COLS]
    _atomic_to_parquet(df, out)
    return df


async def ingest_stocks(timespans=None):
    timespans = timespans or ['day']
    pairs = [(t, ts) for t in STOCK_TICKERS for ts in timespans]

    def _path(t, ts): return DATA_DIR / 'stocks' / f'{t}_{ts}.parquet'

    cached  = [(t, ts) for t, ts in pairs if     _path(t, ts).exists()]
    pending = [(t, ts) for t, ts in pairs if not _path(t, ts).exists()]

    results = {f'{t}_{ts}': pd.read_parquet(_path(t, ts)) for t, ts in cached}

    with tqdm(total=len(pairs), initial=len(cached), desc='Stocks') as pbar:
        for ticker, timespan in pending:
            results[f'{ticker}_{timespan}'] = await fetch_stock_bars(ticker, timespan)
            pbar.update(1)

    return results


In [8]:
stock_data = asyncio.run(ingest_stocks(['day']))

ok = {k: v for k, v in stock_data.items() if not v.empty}
print(f'\nFetched {len(ok)}/{len(stock_data)} tickers')
if ok:
    k0 = next(iter(ok))
    print(f'\nSample -- {k0}:')
    print(ok[k0].tail(3).to_string())

Stocks: 100%|██████████| 45/45 [00:00<?, ?it/s]


Fetched 45/45 tickers

Sample -- AAPL_day:
                          ticker     open    high     low   close        volume      vwap  transactions
ts                                                                                                     
2026-05-28 04:00:00+00:00   AAPL  310.680  312.80  309.57  312.51  4.822039e+07  311.5515        715721
2026-05-29 04:00:00+00:00   AAPL  311.775  315.00  309.53  312.06  7.002675e+07  311.9766        764555
2026-06-01 04:00:00+00:00   AAPL  309.625  310.94  305.02  306.31  4.884993e+07  307.4113        904768


## 2 - Options (historical, trimmed for an overnight pull)

Reconstructs ~2 years of **ATM-ish daily option history** for predictor conditioning — *not* the current chain (those contracts haven't existed long enough to carry history).

1. Enumerate **expired** contracts via `/v3/reference/options/contracts` (`expired=true`), bounded to a strike band around where each underlying actually traded.
2. Locally keep **~monthly expiries** and the **ATM ± N strikes** (spot taken ~30d pre-expiry from the already-pulled stock bars).
3. Pull daily OHLCV per selected contract via `/v2/aggs` — resumable and rail-capped.

> Greeks/IV/OI aren't on the free tier — derive IV later via Black–Scholes inversion from these closes.
> **Rails:** `OPT_MAX_CONTRACTS` (~9h at 5/min) and `OPT_MAX_RUNTIME_H` keep the unattended run inside one night. Re-run any time to resume (existing parquet is skipped).

In [9]:
OPTIONS_UNDERLYINGS = ['AAPL', 'MSFT', 'NVDA', 'SPY', 'QQQ', 'TSLA', 'AMZN', 'GOOGL']

# --- Trimmed HISTORICAL scope (sized for an overnight ~12h free-tier run) ---
# Rebuild ~2 years of ATM-ish option history for conditioning, NOT the current
# chain: enumerate EXPIRED contracts, keep ~monthly expiries and the strikes
# nearest to where the underlying actually traded, then pull daily OHLCV.
OPT_STRIKES_PER_SIDE = 4      # ATM +/- N strikes per expiry per type (2N+1 strikes)
OPT_EXPIRY_STRIDE_D  = 28     # keep roughly one expiry per this many days
OPT_REF_LEAD_DAYS    = 30     # 'ATM' = nearest strikes to spot this many days pre-expiry
OPT_ENUM_MAX_PAGES   = 20     # cap reference enumeration (SPY/QQQ have huge weekly chains)

# --- Hard safety rails for the unattended run ---
OPT_MAX_CONTRACTS = 2800      # cap the per-contract history pull (~9.3h at 5/min)
OPT_MAX_RUNTIME_H = 11.5      # wall-clock stop on the history loop; re-run to resume

print(f'Underlyings: {len(OPTIONS_UNDERLYINGS)}  |  '
      f'ATM +/-{OPT_STRIKES_PER_SIDE} strikes, ~{OPT_EXPIRY_STRIDE_D}d expiry stride')
print(f'Rails: <= {OPT_MAX_CONTRACTS} contracts, <= {OPT_MAX_RUNTIME_H}h history runtime')

Underlyings: 8  |  ATM +/-4 strikes, ~28d expiry stride
Rails: <= 2800 contracts, <= 11.5h history runtime


In [10]:
def _underlying_close(underlying: str) -> Optional[pd.Series]:
    """Naive-indexed daily close series from the already-pulled stock bars."""
    p = DATA_DIR / 'stocks' / f'{underlying}_day.parquet'
    if not p.exists():
        return None
    df = pd.read_parquet(p)
    if df.empty or 'close' not in df.columns:
        return None
    s = df['close'].copy()
    idx = pd.to_datetime(s.index)
    s.index = idx.tz_convert(None) if idx.tz is not None else idx
    return s.sort_index()


async def list_expired_contracts(underlying: str, lo_strike: float, hi_strike: float) -> list:
    """All contracts (incl. expired) expiring in [START, END] within a strike band.

    Most-recent expiries first; capped at OPT_ENUM_MAX_PAGES so a pathological
    weekly chain (SPY/QQQ) can't run away. next_url carries the filters forward.
    """
    url = f'{BASE_URL}/v3/reference/options/contracts'
    params = {
        'underlying_ticker': underlying,
        'expiration_date.gte': START_DATE,
        'expiration_date.lte': END_DATE,
        'strike_price.gte': round(lo_strike, 2),
        'strike_price.lte': round(hi_strike, 2),
        'expired': 'true',
        'sort': 'expiration_date',
        'order': 'desc',
        'limit': 1000,
    }
    rows, first, pages = [], True, 0
    while url and pages < OPT_ENUM_MAX_PAGES:
        data = await client.get(url, params if first else None)
        first = False
        rows.extend(data.get('results') or [])
        url = data.get('next_url', '')
        pages += 1
    if url:
        log.warning('%s: enumeration hit %d-page cap — older expiries truncated',
                    underlying, OPT_ENUM_MAX_PAGES)
    return rows


def select_target_contracts(underlying: str, contracts_df: pd.DataFrame) -> pd.DataFrame:
    """Keep ~monthly expiries and the ATM +/- N strikes per expiry per type."""
    s = _underlying_close(underlying)
    if s is None or s.empty or contracts_df.empty:
        return contracts_df.iloc[0:0]

    df = contracts_df.copy()
    df['expiration_date'] = pd.to_datetime(df['expiration_date'], errors='coerce')
    df['strike_price'] = pd.to_numeric(df['strike_price'], errors='coerce')
    df = df.dropna(subset=['expiration_date', 'strike_price', 'contract_type'])
    if df.empty:
        return df

    # Thin expiries to ~one per OPT_EXPIRY_STRIDE_D days.
    kept, last = [], None
    for e in sorted(df['expiration_date'].unique()):
        e = pd.Timestamp(e)
        if last is None or (e - last).days >= OPT_EXPIRY_STRIDE_D:
            kept.append(e)
            last = e
    df = df[df['expiration_date'].isin(kept)]

    n_keep = OPT_STRIKES_PER_SIDE * 2 + 1
    picks = []
    for (exp, ctype), grp in df.groupby(['expiration_date', 'contract_type']):
        ref_date = pd.Timestamp(exp) - pd.Timedelta(days=OPT_REF_LEAD_DAYS)
        prior = s[s.index <= ref_date]
        spot = prior.iloc[-1] if not prior.empty else s.iloc[0]
        grp = grp.assign(_d=(grp['strike_price'] - spot).abs()).sort_values('_d')
        picks.append(grp.head(n_keep))
    out = pd.concat(picks) if picks else df.iloc[0:0]
    return out.drop(columns='_d', errors='ignore').reset_index(drop=True)


async def ingest_options_contracts() -> dict:
    contracts = {}
    for und in tqdm(OPTIONS_UNDERLYINGS, desc='Option chains (historical)'):
        cache = DATA_DIR / 'options' / 'snapshots' / f'{und}_hist_selected.parquet'
        if cache.exists():
            contracts[und] = pd.read_parquet(cache)
            continue

        s = _underlying_close(und)
        if s is None or s.empty:
            log.warning('No stock bars for %s — run the Stocks cell first; skipping', und)
            contracts[und] = pd.DataFrame()
            continue

        lo, hi = float(s.min()) * 0.7, float(s.max()) * 1.3
        rows = await list_expired_contracts(und, lo, hi)
        allc = pd.DataFrame(rows)
        sel = select_target_contracts(und, allc) if not allc.empty else allc
        if not sel.empty:
            sel['underlying'] = und
        sel.to_parquet(cache, index=False)
        contracts[und] = sel
    return contracts

In [11]:
contracts_data = asyncio.run(ingest_options_contracts())

ok_contracts = {k: v for k, v in contracts_data.items() if not v.empty}
total_sel = sum(len(v) for v in ok_contracts.values())
print(f'\nSelected {total_sel:,} contracts across {len(ok_contracts)}/{len(OPTIONS_UNDERLYINGS)} underlyings')
for und, df in ok_contracts.items():
    n_exp = pd.to_datetime(df['expiration_date']).dt.date.nunique() if 'expiration_date' in df else 0
    print(f'  {und}: {len(df):>4,} contracts  across {n_exp} expiries')
print(f'\n(history pull rail: <= {OPT_MAX_CONTRACTS:,} contracts)')

Option chains (historical): 100%|██████████| 8/8 [00:00<00:00, 734.89it/s]


Selected 2,565 contracts across 8/8 underlyings
  AAPL:  468 contracts  across 26 expiries
  MSFT:  378 contracts  across 21 expiries
  NVDA:  423 contracts  across 24 expiries
  SPY:   54 contracts  across 3 expiries
  QQQ:   63 contracts  across 4 expiries
  TSLA:  243 contracts  across 14 expiries
  AMZN:  468 contracts  across 26 expiries
  GOOGL:  468 contracts  across 26 expiries

(history pull rail: <= 2,800 contracts)


In [12]:
async def fetch_options_bar(contract_ticker: str) -> pd.DataFrame:
    safe = contract_ticker.replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'options' / 'history' / f'{safe}_day.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{contract_ticker}/range'
        f'/1/day/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'false', 'sort': 'asc'})
    if not rows:
        return _empty_ohlcv(contract_ticker, out)

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = contract_ticker
    df = df.set_index('ts')[_OHLCV_COLS]
    # Persist to the filesystem immediately and atomically (write temp + rename),
    # so an interrupt mid-write can't leave a truncated parquet that resume trusts.
    _atomic_to_parquet(df, out)
    return df


async def ingest_options_history(contracts_data: dict) -> dict:
    all_tickers = []
    for df in contracts_data.values():
        if not df.empty and 'ticker' in df.columns:
            all_tickers.extend(df['ticker'].dropna().tolist())
    all_tickers = sorted(set(all_tickers))

    # Budget rail: subsample evenly if the selection exceeds OPT_MAX_CONTRACTS.
    # Deterministic, so a resumed run picks the same subset and reuses saved files.
    if len(all_tickers) > OPT_MAX_CONTRACTS:
        step = len(all_tickers) / OPT_MAX_CONTRACTS
        all_tickers = [all_tickers[int(i * step)] for i in range(OPT_MAX_CONTRACTS)]
        log.info('Capped selection to %d contracts (OPT_MAX_CONTRACTS rail)', len(all_tickers))

    def _path(t):
        safe = t.replace(':', '_').replace('/', '_')
        return DATA_DIR / 'options' / 'history' / f'{safe}_day.parquet'

    # Resume from the filesystem: load every contract already on disk so the run
    # summary reflects the FULL saved corpus, not just this session's fetches.
    # Each new fetch below is written to disk the instant it arrives (in
    # fetch_options_bar), so an interrupt loses at most the one in-flight contract.
    history, pending = {}, []
    for t in all_tickers:
        p = _path(t)
        if p.exists():
            history[t] = pd.read_parquet(p)
        else:
            pending.append(t)
    n_saved = len(history)
    log.info('%d contracts selected — %d already on disk, %d to fetch (~%.1fh at 5/min)',
             len(all_tickers), n_saved, len(pending), len(pending) / 300)

    deadline = time.monotonic() + OPT_MAX_RUNTIME_H * 3600
    stopped = False
    with tqdm(total=len(all_tickers), initial=n_saved, desc='Options history') as pbar:
        for ticker in pending:
            if time.monotonic() >= deadline:
                log.warning('Runtime rail hit (%.1fh) — stopping; re-run to resume',
                            OPT_MAX_RUNTIME_H)
                stopped = True
                break
            history[ticker] = await fetch_options_bar(ticker)  # saved to disk here
            pbar.update(1)

    log.info('Stopped at runtime rail.' if stopped else 'Completed all pending option histories.')
    return history


In [13]:
options_history = asyncio.run(ingest_options_history(contracts_data))

ok_hist = {k: v for k, v in options_history.items() if not v.empty}
total_bars = sum(len(v) for v in ok_hist.values())
print(f'\nHistory: {len(ok_hist)}/{len(options_history)} contracts  |  {total_bars:,} total bars')

10:33:04  INFO      2565 contracts selected — 2565 already on disk, 0 to fetch (~0.0h at 5/min)
Options history: 100%|██████████| 2565/2565 [00:00<?, ?it/s]
10:33:04  INFO      Completed all pending option histories.



History: 2564/2565 contracts  |  94,272 total bars


## 3 - Futures

Pulled from the dedicated **Futures API** (free *Futures Basic* tier: all tickers, 5 req/min, 2yr history, CME/CBOT/NYMEX/COMEX). This is a different path and symbology from stocks/options — continuous `ES1!` symbols do **not** exist here.

1. Enumerate **dated single contracts** per product via `/futures/v1/contracts` (`ESU5` = E-mini S&P, Sep 2025: product code + CME month letter + year digit).
2. Pull daily **session** OHLCV per contract via `/futures/v1/aggs/{ticker}` (`resolution=1session`).

Per-contract parquet, resumable, atomic writes — same rails as the options pull. Futures are predictor **conditioning** (cross-asset signals), not a JEPA training target.

> If a product returns 0 contracts, its `product_code` is wrong for this venue — check the log. If every contract 404s, flip `FUTURES_BASE` to `https://api.massive.com`.

In [14]:
# Product codes (CME Globex roots). If one yields 0 contracts, its code is
# wrong for this venue — the enumeration step logs a warning so you can fix it.
FUTURES_PRODUCTS = [
    'ES', 'NQ', 'RTY', 'YM',     # equity index   (CME)
    'CL', 'NG',                  # energy         (NYMEX)
    'GC', 'SI', 'HG',            # metals         (COMEX)
    'ZB', 'ZN',                  # rates          (CBOT)
    'ZC', 'ZS', 'ZW',            # grains         (CBOT)
]

FUT_ENUM_MAX_PAGES = 10    # cap contract enumeration per product
FUT_MAX_CONTRACTS  = 400   # safety cap on the per-contract history pull

# Futures share the same host/key; flip to 'https://api.massive.com' on 404s.
FUTURES_BASE = BASE_URL

print(f'{len(FUTURES_PRODUCTS)} futures products  |  base {FUTURES_BASE}/futures/v1')

14 futures products  |  base https://api.polygon.io/futures/v1


In [15]:
import json, re

# Outright single-contract ticker = product root + CME month letter + 1-2 year
# digits (ESU5, CLF7). Calendar/inter-commodity spreads ('CL:BZ H7-K7') carry
# ':' / ' ' / '-', slip past type=single (their `type` is null pre-2025-03-12),
# and 500 on the aggs endpoint. Drop them — we condition on outright underlyings.
_MONTH_CODES = 'FGHJKMNQUVXZ'
_OUTRIGHT_RE = re.compile(rf'^[A-Z]{{1,4}}[{_MONTH_CODES}]\d{{1,2}}$')


def _is_outright(ticker: str) -> bool:
    return bool(_OUTRIGHT_RE.match(ticker or ''))


async def list_futures_contracts(product_code: str) -> list:
    """Dated single contracts for a product whose life overlaps [START, END].

    Filter on last_trade_date >= START so we get contracts that were still
    trading at any point in the window (incl. currently-active ones with a
    future last_trade_date). Per-contract aggs below clip to the date range.

    NOTE: /futures/v1/contracts only sorts on {date, product_code, ticker}.
    last_trade_date is filterable but NOT sortable — passing it to `sort`
    returns 400. `date.desc` surfaces most-recently-active contracts first.
    """
    url = f'{FUTURES_BASE}/futures/v1/contracts'
    params = {
        'product_code': product_code,
        'type': 'single',
        'last_trade_date.gte': START_DATE,
        'limit': 1000,
        'sort': 'date.desc',
    }
    rows, first, pages = [], True, 0
    while url and pages < FUT_ENUM_MAX_PAGES:
        try:
            data = await client.get(url, params if first else None)
        except aiohttp.ClientResponseError as exc:
            # Polygon's /contracts endpoint intermittently 500s on deep cursor
            # pages. A flaky page must not abort the whole enumeration — keep
            # what we have for this product (date.desc = newest first) and move on.
            log.warning('%s: enumeration page %d HTTP %s — truncating this product',
                        product_code, pages, exc.status)
            break
        first = False
        rows.extend(data.get('results') or [])
        url = data.get('next_url', '')
        pages += 1
    return rows


async def fetch_futures_contract(ticker: str) -> pd.DataFrame:
    safe = ticker.replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'futures' / f'{safe}_day.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = f'{FUTURES_BASE}/futures/v1/aggs/{ticker}'
    try:
        rows = await client.paginate(url, {
            'resolution': '1session',
            'window_start.gte': START_DATE,
            'window_start.lte': END_DATE,
            'limit': 1000,
            'sort': 'window_start.asc',
        })
    except aiohttp.ClientResponseError as exc:
        # A persistent 5xx on one contract must not abort the whole run. Skip it;
        # don't cache an empty file (a server-side error should retry next run).
        log.warning('%s: HTTP %s after retries — skipping contract', ticker, exc.status)
        return pd.DataFrame(columns=_OHLCV_COLS)
    if not rows:
        return _empty_ohlcv(ticker, out)

    df = pd.DataFrame(rows)
    df['ts'] = pd.to_datetime(df['window_start'], unit='ns', utc=True)  # ns epoch
    df['ticker'] = ticker
    vol = pd.to_numeric(df.get('volume'), errors='coerce')
    dvol = pd.to_numeric(df.get('dollar_volume'), errors='coerce')
    df['vwap'] = (dvol / vol).where(vol > 0)  # futures aggs omit vwap; derive it
    for col in ('open', 'high', 'low', 'close', 'transactions'):
        if col not in df.columns:
            df[col] = pd.NA
    df = df.set_index('ts')[_OHLCV_COLS]
    _atomic_to_parquet(df, out)
    return df


async def ingest_futures() -> dict:
    # 1) Enumerate dated contracts ONCE and cache the ticker list. Enumeration is
    #    ~14 paginated rate-limited calls (~25 min); without this manifest every
    #    re-run repeats it before the (already resumable) history pull. Delete
    #    data/raw/futures/_contracts_manifest.json to force re-enumeration.
    manifest = DATA_DIR / 'futures' / '_contracts_manifest.json'
    if manifest.exists():
        tickers = json.loads(manifest.read_text())
        log.info('Loaded %d futures tickers from manifest — skipping enumeration '
                 '(delete _contracts_manifest.json to refresh)', len(tickers))
    else:
        raw = []
        for prod in tqdm(FUTURES_PRODUCTS, desc='Futures contracts'):
            rows = await list_futures_contracts(prod)
            if not rows:
                log.warning('%s: 0 contracts — wrong product_code for this venue?', prod)
            raw.extend(c['ticker'] for c in rows if c.get('ticker'))

        # Keep only outright single contracts; drop spreads/combos (they 500).
        outrights = sorted({t for t in raw if _is_outright(t)})
        n_dropped = len(set(raw)) - len(outrights)
        if n_dropped:
            log.info('Dropped %d non-outright (spread/combo) tickers', n_dropped)
        tickers = outrights[:FUT_MAX_CONTRACTS]
        if len(outrights) > FUT_MAX_CONTRACTS:
            log.info('Capped to %d futures contracts (FUT_MAX_CONTRACTS rail)', len(tickers))
        if not tickers:
            log.warning('No outright tickers enumerated — NOT writing manifest '
                        '(likely a Polygon 500 outage); re-run to retry')
            return {}
        manifest.write_text(json.dumps(tickers))
        log.info('Wrote futures manifest: %d outright tickers', len(tickers))

    def _path(t):
        safe = t.replace(':', '_').replace('/', '_')
        return DATA_DIR / 'futures' / f'{safe}_day.parquet'

    # 2) Resume from disk, then pull the rest (each saved immediately + atomically).
    history, pending = {}, []
    for t in tickers:
        p = _path(t)
        if p.exists():
            history[t] = pd.read_parquet(p)
        else:
            pending.append(t)
    n_saved = len(history)
    log.info('%d futures contracts — %d on disk, %d to fetch (~%.1f min at 5/min)',
             len(tickers), n_saved, len(pending), len(pending) / 5)

    with tqdm(total=len(tickers), initial=n_saved, desc='Futures history') as pbar:
        for t in pending:
            history[t] = await fetch_futures_contract(t)  # saved to disk here
            pbar.update(1)

    return history

In [16]:
futures_data = asyncio.run(ingest_futures())

ok_fut = {k: v for k, v in futures_data.items() if not v.empty}
total_bars = sum(len(v) for v in ok_fut.values())
print(f'\nFutures: {len(ok_fut)}/{len(futures_data)} contracts  |  {total_bars:,} total bars')
if ok_fut:
    k0 = next(iter(ok_fut))
    print(f'\nSample -- {k0}:')
    print(ok_fut[k0].tail(3).to_string())

10:33:04  INFO      Loaded 400 futures tickers from manifest — skipping enumeration (delete _contracts_manifest.json to refresh)
10:33:04  INFO      400 futures contracts — 7 on disk, 393 to fetch (~78.6 min at 5/min)
Futures history: 100%|██████████| 400/400 [1:28:33<00:00, 13.52s/it]


Futures: 199/400 contracts  |  26,104 total bars

Sample -- CLF7:
                          ticker   open   high    low  close  volume       vwap  transactions
ts                                                                                           
2026-05-31 00:00:00+00:00   CLF7  78.32  80.00  77.97  78.95    1050  79.140133           950
2026-06-01 00:00:00+00:00   CLF7  79.00  79.93  77.79  79.67     766  79.092963           681
2026-06-02 00:00:00+00:00   CLF7  79.65  81.20  79.65  80.67     464  80.593966           409


## 4 - Validation

In [17]:
print('=' * 62)
print('INGESTION SUMMARY')
print('=' * 62)

stock_ok = {k: v for k, v in stock_data.items()     if not v.empty}
chain_ok = {k: v for k, v in contracts_data.items() if not v.empty}
fut_ok   = {k: v for k, v in futures_data.items()   if not v.empty}

# Options history: count files on disk — dict only holds newly-fetched contracts
hist_files   = list((DATA_DIR / 'options' / 'history').glob('*.parquet'))
n_hist       = len(hist_files)
total_hist_bars = sum(len(pd.read_parquet(f)) for f in hist_files)

print(f'\nStocks:            {len(stock_ok):>4} tickers      {sum(len(v) for v in stock_ok.values()):>10,} bars')
print(f'Options contracts: {len(chain_ok):>4} underlyings  {sum(len(v) for v in chain_ok.values()):>10,} contracts')
print(f'Options history:   {n_hist:>4} contracts    {total_hist_bars:>10,} bars  (on disk)')
print(f'Futures:           {len(fut_ok):>4} contracts    {sum(len(v) for v in fut_ok.values()):>10,} bars')

print('\n--- Null % in OHLCV (spot-check first 3 per class) ---')
for label, d in [('Stocks', stock_ok), ('Futures', fut_ok)]:
    for key, df in list(d.items())[:3]:
        cols = [c for c in _OHLCV_COLS[1:] if c in df.columns]
        null_pct = df[cols].isnull().mean().mean() * 100
        date_min = df.index.min().date() if not df.empty else 'n/a'
        date_max = df.index.max().date() if not df.empty else 'n/a'
        print(f'  {label} {key}: {len(df)} rows  {null_pct:.1f}% nulls  [{date_min} -> {date_max}]')

print(f'\nData written to: {DATA_DIR.resolve()}')

INGESTION SUMMARY

Stocks:              45 tickers          22,500 bars
Options contracts:    8 underlyings       2,565 contracts
Options history:   2922 contracts        99,987 bars  (on disk)
Futures:            199 contracts        26,104 bars

--- Null % in OHLCV (spot-check first 3 per class) ---
  Stocks AAPL_day: 500 rows  0.0% nulls  [2024-06-03 -> 2026-06-01]
  Stocks ABBV_day: 500 rows  0.0% nulls  [2024-06-03 -> 2026-06-01]
  Stocks AMD_day: 500 rows  0.0% nulls  [2024-06-03 -> 2026-06-01]
  Futures CLF7: 188 rows  0.0% nulls  [2024-06-10 -> 2026-06-02]
  Futures CLF8: 60 rows  0.0% nulls  [2025-02-12 -> 2026-06-03]
  Futures CLF9: 3 rows  0.0% nulls  [2026-01-04 -> 2026-06-01]

Data written to: /Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/data/raw


In [18]:
asyncio.run(client.close())
print('HTTP session closed.')

HTTP session closed.


In [19]:
import sys
print("kernel python:", sys.executable)
try:
    import certifi, ssl, aiohttp
    print("certifi:", certifi.where())
    print("aiohttp:", aiohttp.__version__)
except Exception as e:
    print("IMPORT FAIL:", repr(e))
# Is the live `client` the patched version?
try:
    print("client has _ssl_ctx attr:", hasattr(client, "_ssl_ctx"))
except NameError:
    print("client not defined yet")


kernel python: /Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/bin/python
certifi: /Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/certifi/cacert.pem
aiohttp: 3.14.0
client has _ssl_ctx attr: True


In [20]:
import asyncio, ssl, certifi, aiohttp

# 1) Prove the certifi context verifies in THIS kernel
async def _probe():
    ctx = ssl.create_default_context(cafile=certifi.where())
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=ctx)) as s:
        async with s.get('https://api.polygon.io/v2/aggs/ticker/AAPL/prev') as r:
            return r.status
print("TLS probe HTTP status:", asyncio.run(_probe()), "(401/403 = cert verified)")

# 2) Hot-patch the live client so you don't have to re-run cells right now
try:
    asyncio.run(client.close())
except Exception:
    pass
client._ssl_ctx = ssl.create_default_context(cafile=certifi.where())
import types
async def _get_session(self):
    if self._session is None or self._session.closed:
        self._session = aiohttp.ClientSession(
            headers={'Authorization': f'Bearer {self.api_key}'},
            timeout=aiohttp.ClientTimeout(total=30),
            connector=aiohttp.TCPConnector(ssl=self._ssl_ctx),
        )
    return self._session
client._get_session = types.MethodType(_get_session, client)
print("live client patched -> has _ssl_ctx:", hasattr(client, "_ssl_ctx"))


TLS probe HTTP status: 401 (401/403 = cert verified)
live client patched -> has _ssl_ctx: True


In [21]:
import asyncio
# Real authenticated call through the patched client (uses your MARKET_DATA_API_KEY)
_data = asyncio.run(client.get(f'{BASE_URL}/v2/aggs/ticker/AAPL/prev'))
print("status field:", _data.get('status'), "| results:", len(_data.get('results') or []))


status field: OK | results: 1


In [22]:
import os, ssl, socket, certifi

# 1) Proxy / TLS-interception env?
for k in ("HTTPS_PROXY","https_proxy","HTTP_PROXY","REQUESTS_CA_BUNDLE","SSL_CERT_FILE","NODE_EXTRA_CA_CERTS"):
    print(f"{k}={os.environ.get(k)}")

# 2) What cert chain does the host actually present, and who issued it?
print("\n--- peer cert via certifi context ---")
ctx = ssl.create_default_context(cafile=certifi.where())
try:
    with socket.create_connection(("api.polygon.io", 443), timeout=10) as sock:
        with ctx.wrap_socket(sock, server_hostname="api.polygon.io") as s:
            cert = s.getpeercert()
            print("VERIFIED OK")
            print("subject:", dict(x[0] for x in cert["subject"]))
            print("issuer :", dict(x[0] for x in cert["issuer"]))
except ssl.SSLCertVerificationError as e:
    print("VERIFY FAILED:", e)

# 3) Pull the raw chain WITHOUT verifying, to see the issuer the server sends
print("\n--- raw chain (no verify) ---")
unv = ssl._create_unverified_context()
with socket.create_connection(("api.polygon.io", 443), timeout=10) as sock:
    with unv.wrap_socket(sock, server_hostname="api.polygon.io") as s:
        der = s.getpeercert(binary_form=True)
        import ssl as _s
        leaf = _s.DER_cert_to_PEM_cert(der)
        c = s.getpeercert()  # empty on unverified, so parse via cryptography if available
print("leaf cert length bytes:", len(der))
try:
    from cryptography import x509
    from cryptography.hazmat.primitives import hashes
    crt = x509.load_der_x509_certificate(der)
    print("leaf subject:", crt.subject.rfc4514_string())
    print("leaf issuer :", crt.issuer.rfc4514_string())
except Exception as e:
    print("cryptography not available:", e)


HTTPS_PROXY=None
https_proxy=None
HTTP_PROXY=None
REQUESTS_CA_BUNDLE=/Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/certifi/cacert.pem
SSL_CERT_FILE=/Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/certifi/cacert.pem
NODE_EXTRA_CA_CERTS=None

--- peer cert via certifi context ---
VERIFIED OK
subject: {'commonName': 'api.polygon.io'}
issuer : {'countryName': 'US', 'organizationName': "Let's Encrypt", 'commonName': 'R12'}

--- raw chain (no verify) ---
leaf cert length bytes: 1296
cryptography not available: No module named 'cryptography'


In [23]:
import ssl, certifi
# What is the CURRENTLY-live client actually using?
print("client type:", type(client).__name__)
print("has _ssl_ctx:", hasattr(client, "_ssl_ctx"))
print("_get_session is patched:", "certifi" in (client._get_session.__doc__ or "") or hasattr(client, "_ssl_ctx"))

# Inspect the class definition that's live in the kernel
import inspect
src = inspect.getsource(type(client).__init__)
print("\n--- live __init__ source ---")
print(src)


client type: PolygonClient
has _ssl_ctx: True
_get_session is patched: True

--- live __init__ source ---
    def __init__(self, api_key: str, rate: float = 5.0):
        self.api_key = api_key
        self._limiter = TokenBucket(rate)
        self._session: Optional[aiohttp.ClientSession] = None
        # Explicit certifi trust store — fixes CERTIFICATE_VERIFY_FAILED on macOS,
        # no-op on Colab/Linux. See CLAUDE.md (notebook env gotcha).
        self._ssl_ctx = ssl.create_default_context(cafile=certifi.where())



In [24]:
import ssl, certifi, aiohttp, asyncio
from typing import Optional

# Tear down any stale session first
try:
    asyncio.run(client.close())
except Exception:
    pass

class PolygonClient:
    _MAX_RETRIES = 5

    def __init__(self, api_key: str, rate: float = 5.0):
        self.api_key = api_key
        self._limiter = TokenBucket(rate)
        self._session: Optional[aiohttp.ClientSession] = None
        self._ssl_ctx = ssl.create_default_context(cafile=certifi.where())

    async def _get_session(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(
                headers={'Authorization': f'Bearer {self.api_key}'},
                timeout=aiohttp.ClientTimeout(total=30),
                connector=aiohttp.TCPConnector(ssl=self._ssl_ctx),
            )
        return self._session

    async def close(self) -> None:
        if self._session and not self._session.closed:
            await self._session.close()

    async def get(self, url: str, params: Optional[dict] = None) -> dict:
        session = await self._get_session()
        for attempt in range(self._MAX_RETRIES):
            await self._limiter.acquire()
            try:
                async with session.get(url, params=params) as resp:
                    if resp.status == 429:
                        log.warning('429 rate-limit; sleeping 61s')
                        await asyncio.sleep(61)
                        continue
                    resp.raise_for_status()
                    return await resp.json()
            except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                if attempt == self._MAX_RETRIES - 1:
                    raise
                wait = 2 ** attempt
                log.warning('Error (attempt %d): %s -- retry in %ds', attempt + 1, exc, wait)
                await asyncio.sleep(wait)
        return {}

    async def paginate(self, url: str, params: Optional[dict] = None) -> list:
        all_results = []
        first = True
        while url:
            data = await self.get(url, params if first else None)
            first = False
            all_results.extend(data.get('results') or [])
            url = data.get('next_url', '')
        return all_results

client = PolygonClient(API_KEY, RATE_LIMIT)
print("client rebuilt | has _ssl_ctx:", hasattr(client, "_ssl_ctx"))

# Prove an authenticated fetch works end-to-end now
_d = asyncio.run(client.get(f'{BASE_URL}/v2/aggs/ticker/AAPL/prev'))
print("auth fetch status:", _d.get('status'), "| results:", len(_d.get('results') or []))


client rebuilt | has _ssl_ctx: True
auth fetch status: OK | results: 1


In [25]:
import os, ssl, certifi, asyncio, aiohttp

# Point OpenSSL's DEFAULT context at certifi via env var — this makes even the
# stale-buffer code (plain ClientSession, no connector) verify correctly.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
print("SSL_CERT_FILE ->", os.environ["SSL_CERT_FILE"])

# Prove the DEFAULT context (what the old code uses) now verifies
async def _probe():
    async with aiohttp.ClientSession() as s:   # no connector => default ctx
        async with s.get("https://api.polygon.io/v2/aggs/ticker/AAPL/prev") as r:
            return r.status
print("default-context TLS status:", asyncio.run(_probe()), "(401 = verified)")

# Rebuild the live client too so the current run works immediately
try:
    asyncio.run(client.close())
except Exception:
    pass
client = PolygonClient(API_KEY, RATE_LIMIT)
_d = asyncio.run(client.get(f"{BASE_URL}/v2/aggs/ticker/AAPL/prev"))
print("client auth fetch:", _d.get("status"), "| results:", len(_d.get("results") or []))


SSL_CERT_FILE -> /Users/shreyasnatarajan/Desktop/Personal_projects/JEPA-quant/.venv/lib/python3.11/site-packages/certifi/cacert.pem
default-context TLS status: 401 (401 = verified)
client auth fetch: OK | results: 1


In [26]:
import os, asyncio, aiohttp, truststore

# Clear the env-var experiment so we test truststore in isolation
os.environ.pop("SSL_CERT_FILE", None)
os.environ.pop("REQUESTS_CA_BUNDLE", None)

# Patch ssl globally to use the OS-native verifier (does AIA intermediate fetch)
truststore.inject_into_ssl()

# Hammer it: 12 fresh connections to hit multiple CDN edge nodes
async def _hammer(n=12):
    ok = bad = 0
    async with aiohttp.ClientSession() as s:  # default ctx now = truststore
        for _ in range(n):
            try:
                async with s.get("https://api.polygon.io/v2/aggs/ticker/AAPL/prev") as r:
                    ok += 1
            except Exception as e:
                bad += 1
                print("  fail:", type(e).__name__)
    return ok, bad

ok, bad = asyncio.run(_hammer())
print(f"truststore result: {ok} ok / {bad} failed out of 12")


truststore result: 12 ok / 0 failed out of 12


In [27]:
import socket, ssl, certifi, truststore

host = "api.polygon.io"
# 1) How many edge IPs?
ips = sorted({ai[4][0] for ai in socket.getaddrinfo(host, 443, proto=socket.IPPROTO_TCP)})
print("resolved IPs:", ips)

# 2) RAW issuer presented NOW (unverified) — Let's Encrypt = real, else = interception
def raw_issuer(ip):
    ctx = ssl._create_unverified_context()
    with socket.create_connection((ip, 443), timeout=8) as s:
        with ctx.wrap_socket(s, server_hostname=host) as ss:
            der = ss.getpeercert(binary_form=True)
    # crude issuer extraction without cryptography: use ssl's DER->dict via a verified-ish parse
    import ssl as _ssl
    tmp = _ssl._ssl._test_decode_cert if hasattr(_ssl._ssl, "_test_decode_cert") else None
    return len(der)

for ip in ips:
    try:
        print(f"  {ip}: raw leaf bytes={raw_issuer(ip)}")
    except Exception as e:
        print(f"  {ip}: {type(e).__name__}: {e}")

# 3) Direct-socket verify with certifi (regressed?) and with OS verifier (truststore)
for label, ctx in [("certifi", ssl.create_default_context(cafile=certifi.where())),
                    ("os-truststore", truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT))]:
    try:
        with socket.create_connection((host, 443), timeout=8) as s:
            with ctx.wrap_socket(s, server_hostname=host) as ss:
                c = ss.getpeercert()
                print(f"{label}: VERIFIED  issuer={dict(x[0] for x in c['issuer']).get('commonName') if c else '?'}")
    except Exception as e:
        print(f"{label}: FAIL {type(e).__name__}: {e}")


resolved IPs: ['198.44.194.168']
  198.44.194.168: raw leaf bytes=1296
certifi: VERIFIED  issuer=R12
os-truststore: VERIFIED  issuer=R12


In [28]:
import ssl, certifi, aiohttp, asyncio

# Rebuild the live client with the EXPLICIT certifi connector (the committed fix)
try:
    asyncio.run(client.close())
except Exception:
    pass
client = PolygonClient(API_KEY, RATE_LIMIT)
print("client has _ssl_ctx:", hasattr(client, "_ssl_ctx"))

# Hammer aiohttp WITH the certifi connector to prove it's reliable, not flaky
async def _hammer(n=10):
    ctx = ssl.create_default_context(cafile=certifi.where())
    ok = bad = 0
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=ctx)) as s:
        for _ in range(n):
            try:
                async with s.get("https://api.polygon.io/v2/aggs/ticker/AAPL/prev") as r:
                    ok += 1
            except Exception as e:
                bad += 1; print("  fail:", type(e).__name__)
    return ok, bad
ok, bad = asyncio.run(_hammer())
print(f"aiohttp+certifi-connector: {ok} ok / {bad} fail")

# And a real authenticated fetch through the rebuilt client
_d = asyncio.run(client.get(f"{BASE_URL}/v2/aggs/ticker/AAPL/prev"))
print("client auth fetch:", _d.get("status"), "| results:", len(_d.get("results") or []))


client has _ssl_ctx: True
aiohttp+certifi-connector: 10 ok / 0 fail
client auth fetch: OK | results: 1
